In [0]:
import os
import re
import json
import yaml
from urllib.parse import urlparse

# ============================================================
# Silver entity contracts: the silver data model (decided by humans, versioned in Git).
# One YAML file per entity in domain_model/entities/.
# Agents never change contracts: they write mappings (bronze -> entity) that must comply with them.
# Each contract produces ONE silver table:
#   entity     records as delivered by each data source: PK = business keys + source_name
#              (several sources can coexist; resolving them into a golden record is the job of MDM, later)
#   reference  reference data: PK = business keys. Built from a csv in reference_data/ (reference_file)
#              or, for data_source, from the bronze source specs (reference_source: source_specs)
# Every table has FKDataSource -> data_source.PKDataSource:
#   entities: md5(source_name) per row · reference tables: their reference file (or 'source_specs').
# Other references (FKs) can only point to reference entities for now; references between entities
# will point to the MDM golden records (<entity>_golden).
# Silver keeps the resolution published by the source (e.g. hourly and quarter-hourly prices);
# aggregating to a common resolution is a gold decision.
# Naming: "source" always means data source. Energy technologies are a business column called 'technology'.
# Keys: first column PK<Table> = md5(concat_ws("|", key columns)); references are FK<Table>.
# Requires: common/_utils_config and common/_utils_reference_data (%run before this notebook)
# Cells: 1 constants/naming · 2 load · 3 validation · 4 derived tables · 5 ERD (Mermaid) · 6 docs
# ============================================================

entity_contracts_folder = "domain_model/entities"
source_specs_folder = "source_specs"
reference_data_folder = "reference_data"
reference_sources_file = "reference_sources.yml"
docs_folder = "docs"
data_model_doc_name = "silver_data_model.md"

allowed_table_types = {"entity", "reference"}
allowed_column_types = {"string", "int", "bigint", "double", "boolean", "date", "timestamp", "array<string>"}
numeric_column_types = {"int", "bigint", "double"}
invalid_key_types = {"double", "boolean", "array<string>"}
decimal_type_pattern = r"decimal\((\d{1,2}),(\d{1,2})\)"
allowed_completeness_periods = {"day", "month"}
golden_suffix = "_golden"            # reserved for the future MDM golden record tables

# The data_source dimension
data_source_entity = "data_source"
platform_source_name = "source_specs"            # data_source row describing the catalogue itself
allowed_reference_sources = {"source_specs"}
data_source_fields = ["source_name", "source_type", "provider", "description", "license", "base_url", "status"]

allowed_contract_keys = {"entity", "description", "domain", "table_type", "reference_file", "reference_source",
                         "grain", "business_keys", "references", "columns", "completeness"}
allowed_column_keys = {"name", "type", "description", "nullable", "unit", "accepted_values", "range"}
allowed_completeness_keys = {"time_column", "period", "group_by", "expected_rows", "resolution_column"}

# Columns added by the framework (contracts cannot define them)
source_columns = [("source_name", "string"), ("source_endpoint", "string"), ("source_record_id", "string")]
lineage_columns = [("bronze_table", "string"), ("source_file", "string"), ("chunk_start", "date"),
                   ("chunk_end", "date"), ("bronze_run_id", "string"), ("silver_run_id", "string"),
                   ("processed_at", "timestamp")]
entity_reserved_columns = {name for name, _ in source_columns + lineage_columns}   # entities get both
reference_reserved_columns = {name for name, _ in lineage_columns}                # reference tables: lineage only
erd_hidden_columns = {"source_endpoint", "source_record_id"} | reference_reserved_columns


def to_camel_case(snake_name):
    return "".join(part.capitalize() for part in snake_name.split("_"))


def pk_column_name(table_name):
    return f"PK{to_camel_case(table_name)}"


def fk_column_name(table_name):
    return f"FK{to_camel_case(table_name)}"


def silver_table_full_name(table_name):
    return f"{catalog}.{silver_schema}.{table_name}"


def find_repo_folder(folder_name):
    """Walk up from the running notebook's folder until folder_name (e.g. 'domain_model/entities') is found."""
    current = os.getcwd()
    while True:
        candidate = os.path.join(current, folder_name)
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(f"Folder '{folder_name}' not found from {os.getcwd()} upwards")
        current = parent

In [0]:
# ------------------------------------------------------------
# Load contracts, bronze source specs and reference file metadata (YAML files in the repo)
# ------------------------------------------------------------
def read_yaml_file(path):
    with open(path, encoding="utf-8") as f:
        try:
            return yaml.safe_load(f.read())
        except yaml.YAMLError as e:
            raise ValueError(f"{path}: invalid YAML: {str(e)[:200]}")


def load_entity_contract(yaml_text):
    try:
        contract = yaml.safe_load(yaml_text)
    except yaml.YAMLError as e:
        raise ValueError(f"Invalid YAML: {str(e)[:300]}")
    if not isinstance(contract, dict):
        raise ValueError("YAML contract must be a mapping (key: value) at the top level")
    return contract


def list_entity_contracts():
    folder = find_repo_folder(entity_contracts_folder)
    return sorted(f[:-4] for f in os.listdir(folder) if f.endswith(".yml"))


def load_entity_contract_file(entity_name):
    path = os.path.join(find_repo_folder(entity_contracts_folder), f"{entity_name}.yml")
    with open(path, encoding="utf-8") as f:
        contract = load_entity_contract(f.read())
    if contract.get("entity") != entity_name:
        raise ValueError(f"{path}: entity '{contract.get('entity')}' must match the file name")
    return contract


def load_all_entity_contracts():
    """All contracts by entity name."""
    return {name: load_entity_contract_file(name) for name in list_entity_contracts()}


def list_bronze_source_specs():
    """Bronze source specs (source_specs/*.yml) as dicts."""
    folder = find_repo_folder(source_specs_folder)
    return [read_yaml_file(os.path.join(folder, f)) for f in sorted(os.listdir(folder)) if f.endswith(".yml")]


def read_reference_sources():
    """reference_data/reference_sources.yml: {<file>.csv: {description, provider, license}}."""
    path = os.path.join(find_repo_folder(reference_data_folder), reference_sources_file)
    if not os.path.exists(path):
        return {}
    data = read_yaml_file(path) or {}
    if not isinstance(data, dict):
        raise ValueError(f"{path} must be a mapping: <file>.csv -> metadata")
    return data

In [0]:
# ------------------------------------------------------------
# Validation (pure logic; reference contracts also read their csv / the source specs)
# ------------------------------------------------------------
def is_decimal_type(column_type):
    return isinstance(column_type, str) and re.fullmatch(decimal_type_pattern, column_type) is not None


def is_valid_column_type(column_type):
    if is_decimal_type(column_type):
        precision, scale = map(int, re.fullmatch(decimal_type_pattern, column_type).groups())
        return 1 <= precision <= 38 and scale <= precision
    return column_type in allowed_column_types


def is_numeric_type(column_type):
    return column_type in numeric_column_types or is_decimal_type(column_type)


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def contract_columns(contract):
    """Columns of a contract by name (ignores malformed entries; validation reports them)."""
    return {c["name"]: c for c in contract.get("columns") or [] if isinstance(c, dict) and "name" in c}


def validate_contract_columns(contract, reserved_columns):
    columns = contract.get("columns")
    if not isinstance(columns, list) or not columns:
        return ["columns must be a non-empty list"]

    errors, seen = [], set()
    for i, column in enumerate(columns):
        if not isinstance(column, dict):
            errors.append(f"column #{i} must be a mapping")
            continue
        name = str(column.get("name", ""))
        label = name or f"#{i}"
        column_type = column.get("type")

        if not re.fullmatch(r"[a-z][a-z0-9_]{1,60}", name):
            errors.append(f"column {label}: name must be snake_case")
        if name in seen:
            errors.append(f"column {label}: duplicated name")
        seen.add(name)
        if name in reserved_columns:
            errors.append(f"column {label}: '{name}' is added by the framework; remove it from the contract")
        if not is_valid_column_type(column_type):
            errors.append(f"column {label}: type must be one of {sorted(allowed_column_types)} or decimal(p,s)")
        if not str(column.get("description", "")).strip():
            errors.append(f"column {label}: description is required")
        if "nullable" in column and not isinstance(column["nullable"], bool):
            errors.append(f"column {label}: nullable must be true or false")

        if "accepted_values" in column:
            values = column["accepted_values"]
            if column_type != "string":
                errors.append(f"column {label}: accepted_values only applies to string columns")
            elif not isinstance(values, list) or not values or not all(isinstance(v, str) for v in values):
                errors.append(f"column {label}: accepted_values must be a non-empty list of strings "
                              f"(quote numeric codes)")

        if "range" in column:
            value_range = column["range"]
            if not is_numeric_type(column_type):
                errors.append(f"column {label}: range only applies to numeric columns")
            elif (not isinstance(value_range, dict) or not value_range or set(value_range) - {"min", "max"}
                  or not all(is_number(v) for v in value_range.values())):
                errors.append(f"column {label}: range must be a mapping with min and/or max numbers")
            elif "min" in value_range and "max" in value_range and value_range["min"] >= value_range["max"]:
                errors.append(f"column {label}: range min must be lower than max")

        unknown = set(column) - allowed_column_keys
        if unknown:
            errors.append(f"column {label}: unknown keys {sorted(unknown)}")
    return errors


def validate_business_keys(contract, columns):
    keys = contract.get("business_keys")
    if not isinstance(keys, list) or not keys:
        return ["business_keys must be a non-empty list of column names"]
    errors = []
    if len(set(keys)) != len(keys):
        errors.append("business_keys has duplicated columns")
    for key in keys:
        if key not in columns:
            errors.append(f"business key '{key}' is not a column")
        elif columns[key].get("nullable") is True:
            errors.append(f"business key '{key}' cannot be nullable")
        elif columns[key].get("type") in invalid_key_types:
            errors.append(f"business key '{key}' cannot be of type {columns[key]['type']} "
                          f"(use string, int, bigint, date, timestamp or decimal)")
    return errors


def validate_references(contract, columns, contracts):
    references = contract.get("references", [])
    if not isinstance(references, list):
        return ["references must be a list"]
    errors, seen_targets = [], set()
    for i, reference in enumerate(references):
        if not isinstance(reference, dict):
            errors.append(f"reference #{i} must be a mapping")
            continue
        target = reference.get("entity")
        reference_columns = reference.get("columns")
        label = f"reference {target or '#' + str(i)}"

        if target == contract.get("entity"):
            errors.append(f"{label}: an entity cannot reference itself")
            continue
        if target == data_source_entity:
            errors.append(f"{label}: {fk_column_name(data_source_entity)} is added automatically to every table; "
                          f"remove this reference")
            continue
        if target not in contracts:
            errors.append(f"{label}: entity '{target}' has no contract in {entity_contracts_folder}/")
            continue
        if contracts[target].get("table_type") != "reference":
            errors.append(f"{label}: references can only point to reference entities for now "
                          f"(references between entities will point to MDM golden records)")
            continue
        if target in seen_targets:
            errors.append(f"{label}: duplicated reference (one FK per referenced entity)")
        seen_targets.add(target)

        target_keys = contracts[target].get("business_keys") or []
        target_columns = contract_columns(contracts[target])
        if not isinstance(reference_columns, list) or len(reference_columns) != len(target_keys):
            errors.append(f"{label}: columns {reference_columns} must match the business keys of "
                          f"{target} {target_keys} (same number and order)")
            continue
        for own, theirs in zip(reference_columns, target_keys):
            if own not in columns:
                errors.append(f"{label}: '{own}' is not a column of this entity")
                continue
            own_type = columns[own].get("type")
            their_type = target_columns.get(theirs, {}).get("type")
            if own_type != their_type:
                errors.append(f"{label}: column '{own}' ({own_type}) does not match the type of "
                              f"{target}.{theirs} ({their_type})")
        unknown = set(reference) - {"entity", "columns"}
        if unknown:
            errors.append(f"{label}: unknown keys {sorted(unknown)}")
    return errors


def validate_completeness(contract, columns):
    """Expected rows per period: fixed (expected_rows) or derived from a resolution column (1440 / minutes)."""
    completeness = contract.get("completeness")
    if completeness is None:
        return []
    if contract.get("table_type") == "reference":
        return ["completeness only applies to entities of table_type 'entity'"]
    if not isinstance(completeness, dict):
        return ["completeness must be a mapping"]
    errors = []
    time_column = completeness.get("time_column")
    if time_column not in columns or columns[time_column].get("type") not in ("timestamp", "date"):
        errors.append("completeness.time_column must be a timestamp or date column")
    elif time_column not in (contract.get("business_keys") or []):
        errors.append("completeness.time_column must be one of the business_keys")
    if completeness.get("period") not in allowed_completeness_periods:
        errors.append(f"completeness.period must be one of {sorted(allowed_completeness_periods)}")
    group_by = completeness.get("group_by", [])
    if not isinstance(group_by, list) or any(c not in columns for c in group_by):
        errors.append("completeness.group_by must be a list of columns of this entity")

    has_expected, has_resolution = "expected_rows" in completeness, "resolution_column" in completeness
    if has_expected == has_resolution:
        errors.append("completeness needs exactly one of expected_rows (fixed rows per period) or "
                      "resolution_column (rows per day = 1440 / resolution minutes)")
    elif has_expected:
        expected = completeness["expected_rows"]
        if not isinstance(expected, int) or isinstance(expected, bool) or expected < 1:
            errors.append("completeness.expected_rows must be a positive integer")
    else:
        resolution_column = completeness["resolution_column"]
        if resolution_column not in columns or columns[resolution_column].get("type") not in ("int", "bigint"):
            errors.append("completeness.resolution_column must be an int or bigint column of this entity "
                          "(minutes per row)")
        elif columns[resolution_column].get("nullable") is not False:
            errors.append("completeness.resolution_column must be declared with nullable: false")
        if completeness.get("period") != "day":
            errors.append("completeness.resolution_column requires period: day (1440 minutes)")

    unknown = set(completeness) - allowed_completeness_keys
    if unknown:
        errors.append(f"completeness: unknown keys {sorted(unknown)}")
    return errors


def validate_data_source_contract(contract, columns, contracts):
    """The data_source entity: built from the bronze source specs, keyed by source_name."""
    errors = []
    if contract.get("reference_source") not in allowed_reference_sources:
        errors.append(f"reference_source must be one of {sorted(allowed_reference_sources)}")
    if contract.get("entity") != data_source_entity:
        errors.append(f"reference_source is only used by the '{data_source_entity}' entity")
    if contract.get("business_keys") != ["source_name"]:
        errors.append(f"{data_source_entity} business_keys must be [source_name] "
                      f"(every table references it with md5(source_name))")
    unknown = [c for c in columns if c not in data_source_fields]
    if unknown:
        errors.append(f"columns {unknown} are not data_source fields; available: {data_source_fields}")
    try:
        names = [r["source_name"] for r in build_data_source_rows(contracts)]
        duplicated = sorted({n for n in names if names.count(n) > 1})
        if duplicated:
            errors.append(f"duplicated data source names {duplicated} "
                          f"(bronze source specs and reference files must not share names)")
    except Exception as e:
        errors.append(f"cannot build the data_source rows: {str(e)[:200]}")
    return errors


def validate_reference_origin(contract, columns, contracts):
    """Reference entities come from exactly one origin: a csv (reference_file) or the source specs."""
    has_file, has_source = "reference_file" in contract, "reference_source" in contract
    if contract.get("table_type") != "reference":
        return [f"{key} only applies to reference entities"
                for key in ("reference_file", "reference_source") if key in contract]
    if has_file == has_source:
        return ["reference entities need exactly one of reference_file (a csv in reference_data/) "
                "or reference_source (source_specs)"]
    if has_source:
        return validate_data_source_contract(contract, columns, contracts)

    reference_file = str(contract.get("reference_file", ""))
    if not re.fullmatch(r"[A-Za-z0-9_\-]+\.csv", reference_file):
        return ["reference_file must be a csv file name in reference_data/ (no paths)"]
    try:
        rows = read_reference_csv(reference_file)
    except Exception as e:
        return [f"cannot read reference_data/{reference_file}: {str(e)[:150]}"]
    if not rows:
        return [f"reference_data/{reference_file} is empty"]
    errors = []
    missing = [c for c in columns if c not in rows[0]]
    if missing:
        errors.append(f"reference_data/{reference_file} has no columns {missing} "
                      f"(column names must match the csv header {list(rows[0].keys())})")
    try:
        metadata = read_reference_sources().get(reference_file)
    except Exception as e:
        return errors + [f"cannot read {reference_data_folder}/{reference_sources_file}: {str(e)[:150]}"]
    if (not isinstance(metadata, dict) or not str(metadata.get("description", "")).strip()
            or not str(metadata.get("provider", "")).strip()):
        errors.append(f"{reference_data_folder}/{reference_sources_file} needs an entry for {reference_file} "
                      f"with description and provider (license recommended)")
    return errors


def validate_entity_contract(contract, contracts):
    """contracts: all contracts by entity name (needed to check references)."""
    if not isinstance(contract, dict):
        return ["contract must be a mapping (key: value)"]
    errors = []
    entity = str(contract.get("entity", ""))
    table_type = contract.get("table_type")
    if not re.fullmatch(r"[a-z][a-z0-9_]{2,40}", entity):
        errors.append("entity must be snake_case, 3-41 chars")
    elif entity.endswith(golden_suffix):
        errors.append(f"entity cannot end with '{golden_suffix}' (reserved for MDM golden record tables)")
    if entity == data_source_entity and (table_type != "reference" or "reference_source" not in contract):
        errors.append(f"the '{data_source_entity}' entity must be a reference entity with "
                      f"reference_source: source_specs")
    for field in ("description", "domain", "grain"):
        if not str(contract.get(field, "")).strip():
            errors.append(f"{field} is required")
    if table_type not in allowed_table_types:
        errors.append(f"table_type must be one of {sorted(allowed_table_types)}")
    unknown = set(contract) - allowed_contract_keys
    if unknown:
        errors.append(f"unknown keys {sorted(unknown)}")

    reserved = reference_reserved_columns if table_type == "reference" else entity_reserved_columns
    errors += validate_contract_columns(contract, reserved)
    columns = contract_columns(contract)
    errors += validate_business_keys(contract, columns)
    errors += validate_references(contract, columns, contracts)
    errors += validate_completeness(contract, columns)
    errors += validate_reference_origin(contract, columns, contracts)
    return errors


def validate_all_entity_contracts(contracts=None):
    """Returns {entity: [errors]} only for contracts with errors ('(model)' for model-level errors)."""
    contracts = contracts if contracts is not None else load_all_entity_contracts()
    results = {}
    if contracts and data_source_entity not in contracts:
        results["(model)"] = [f"a '{data_source_entity}' contract is required: every silver table references it"]
    for name, contract in contracts.items():
        errors = validate_entity_contract(contract, contracts)
        if errors:
            results[name] = errors
    return results

In [0]:
# ------------------------------------------------------------
# Physical silver tables derived from a contract, and rows of the data_source table (pure logic, no Spark).
# Column order: PK, FKDataSource, other FKs, contract columns, source columns (entities only), lineage.
# build_silver_tables returns a list so that future derived tables (e.g. MDM golden records) fit in.
# ------------------------------------------------------------
def reference_file_source_name(reference_file):
    """Data source name of a reference file: locations_es.csv -> locations_es."""
    return re.sub(r"\.csv$", "", reference_file)


def table_data_source(contract):
    """Fixed data source of a reference table; None for entities (one data source per row)."""
    if contract.get("reference_source"):
        return platform_source_name
    if contract.get("reference_file"):
        return reference_file_source_name(contract["reference_file"])
    return None


def build_silver_tables(contract):
    entity = contract["entity"]
    is_reference = contract["table_type"] == "reference"
    pk_columns = contract["business_keys"] + ([] if is_reference else ["source_name"])
    data_source_fk = {
        "name": fk_column_name(data_source_entity),
        "columns": [] if is_reference else ["source_name"],
        "constant": table_data_source(contract),
        "references_table": data_source_entity,
        "references_column": pk_column_name(data_source_entity),
    }
    contract_fks = [{"name": fk_column_name(r["entity"]), "columns": r["columns"], "constant": None,
                     "references_table": r["entity"], "references_column": pk_column_name(r["entity"])}
                    for r in contract.get("references", [])]
    foreign_keys = [data_source_fk] + contract_fks
    table = {
        "name": entity,
        "entity": entity,
        "role": "reference" if is_reference else "entity",
        "primary_key": {"name": pk_column_name(entity), "columns": pk_columns},
        "foreign_keys": foreign_keys,
        "columns": ([(pk_column_name(entity), "string")]
                    + [(fk["name"], "string") for fk in foreign_keys]
                    + [(c["name"], c["type"]) for c in contract["columns"]]
                    + ([] if is_reference else source_columns)
                    + lineage_columns),
    }
    return [table]


def fk_hash_text(fk):
    if fk["constant"]:
        return f"md5('{fk['constant']}')"
    return f"md5({', '.join(fk['columns'])})"


def describe_silver_tables(contract):
    """Readable summary of the tables a contract creates (for human review)."""
    lines = []
    for table in build_silver_tables(contract):
        pk = table["primary_key"]
        lines.append(f"{silver_table_full_name(table['name'])}  ({table['role']})")
        lines.append(f"  PK {pk['name']} = md5({', '.join(pk['columns'])})")
        for fk in table["foreign_keys"]:
            lines.append(f"  FK {fk['name']} = {fk_hash_text(fk)} -> {fk['references_table']}.{fk['references_column']}")
        lines.append("  columns: " + ", ".join(f"{name} {column_type}" for name, column_type in table["columns"]))
    return "\n".join(lines)


def base_url(url):
    parsed = urlparse(str(url))
    return f"{parsed.scheme}://{parsed.hostname}" if parsed.hostname else str(url)


def build_data_source_rows(contracts):
    """Rows of the data_source table: bronze source specs + reference files + the platform metadata row."""
    rows = []
    for spec in list_bronze_source_specs():
        if not isinstance(spec, dict) or not spec.get("source_name"):
            continue
        urls = sorted({base_url(ep["url"]) for ep in spec.get("endpoints") or []
                       if isinstance(ep, dict) and ep.get("url")})
        rows.append({"source_name": spec["source_name"], "source_type": spec.get("source_type", "http_api"),
                     "provider": spec.get("provider"), "description": spec.get("description"),
                     "license": spec.get("license"), "base_url": ", ".join(urls), "status": "active"})

    reference_sources = read_reference_sources()
    reference_files = sorted({c["reference_file"] for c in contracts.values()
                              if c.get("table_type") == "reference" and c.get("reference_file")})
    for reference_file in reference_files:
        metadata = reference_sources.get(reference_file) or {}
        rows.append({"source_name": reference_file_source_name(reference_file), "source_type": "reference_file",
                     "provider": metadata.get("provider"), "description": metadata.get("description"),
                     "license": metadata.get("license"), "base_url": f"{reference_data_folder}/{reference_file}",
                     "status": "active"})

    rows.append({"source_name": platform_source_name, "source_type": "platform_metadata",
                 "provider": "Data platform (internal)",
                 "description": "Catalogue of data sources, built from the bronze source specs",
                 "license": "internal", "base_url": f"{source_specs_folder}/", "status": "active"})
    return rows

In [0]:
# ------------------------------------------------------------
# ERD: Mermaid erDiagram generated from the contracts (deterministic, never written by the LLM)
# Colour = table role (reference / entity); new tables get a thick dashed border.
# The self-reference of data_source (to its 'source_specs' row) is not drawn.
# ------------------------------------------------------------
erd_role_styles = {
    "reference": {"style": "fill:#EEEDFE,stroke:#534AB7,color:#26215C", "label": "Reference / dimension"},
    "entity":    {"style": "fill:#E1F5EE,stroke:#0F6E56,color:#04342C", "label": "Entity (records by source)"},
}
erd_new_table_style = "stroke-width:3px,stroke-dasharray:6 3"


def mermaid_type(column_type):
    """Mermaid attribute types only accept letters, digits and _ (decimal(10,2) -> decimal_10_2)."""
    return re.sub(r"[^A-Za-z0-9_]+", "_", column_type).strip("_")


def build_erd_mermaid(contracts, focus=None, highlight=(), show_lineage=False):
    """focus: entity names shown with their direct neighbours (None = whole model).
    highlight: entity names whose tables are marked as new (e.g. the proposal under review)."""
    tables = {t["name"]: t for c in contracts.values() for t in build_silver_tables(c)}

    if focus:
        focus_tables = {name for name, t in tables.items() if t["entity"] in focus}
        shown = set(focus_tables)
        for name, table in tables.items():
            for fk in table["foreign_keys"]:
                if name in focus_tables:
                    shown.add(fk["references_table"])
                if fk["references_table"] in focus_tables:
                    shown.add(name)
        shown &= set(tables)
    else:
        shown = set(tables)

    lines = ["erDiagram"]
    for name in sorted(shown):
        for fk in tables[name]["foreign_keys"]:
            target = fk["references_table"]
            if target in shown and target != name:
                lines.append(f'  {target} ||--o{{ {name} : "{fk["name"]}"')

    hidden = set() if show_lineage else erd_hidden_columns
    for name in sorted(shown):
        table = tables[name]
        fk_names = {fk["name"] for fk in table["foreign_keys"]}
        lines.append(f"  {name} {{")
        for column, column_type in table["columns"]:
            if column in hidden:
                continue
            marker = " PK" if column == table["primary_key"]["name"] else (" FK" if column in fk_names else "")
            lines.append(f"    {mermaid_type(column_type)} {column}{marker}")
        lines.append("  }")

    # One style class per (role, new) combination: each table gets exactly one class
    style_groups = {}
    for name in sorted(shown):
        role = tables[name]["role"]
        is_new = tables[name]["entity"] in highlight
        style_groups.setdefault((role, is_new), []).append(name)
    for (role, is_new), names in sorted(style_groups.items()):
        class_name = f"{role}_new" if is_new else role
        style = erd_role_styles[role]["style"] + (f",{erd_new_table_style}" if is_new else "")
        lines.append(f"  classDef {class_name} {style}")
        lines.append(f"  class {','.join(names)} {class_name}")
    return "\n".join(lines)


def erd_legend_html(mermaid_text):
    """Small legend with the roles present in the diagram (and 'new' if any table is highlighted)."""
    items = []
    for role, info in erd_role_styles.items():
        if re.search(rf"classDef {role}(_new)? ", mermaid_text):
            fill, stroke = re.search(r"fill:(#\w+),stroke:(#\w+)", info["style"]).groups()
            items.append(f'<span style="display:inline-flex;align-items:center;gap:6px;margin-right:16px">'
                         f'<span style="width:14px;height:14px;background:{fill};border:1px solid {stroke};'
                         f'border-radius:3px"></span>{info["label"]}</span>')
    if "_new " in mermaid_text:
        items.append('<span style="display:inline-flex;align-items:center;gap:6px">'
                     '<span style="width:14px;height:14px;border:3px dashed #5F5E5A;border-radius:3px"></span>'
                     'New (proposal under review)</span>')
    return f'<div style="font-family:sans-serif;font-size:13px;margin-top:8px">{"".join(items)}</div>'


def display_erd(mermaid_text):
    """Renders a Mermaid diagram in the notebook. Mermaid is loaded by the browser from a CDN.
    If the Mermaid version does not support styling (classDef), it retries without it; if it still
    fails, it shows the diagram source."""
    html = f"""
<div id="erd"></div>
<pre id="erd-fallback" style="display:none;white-space:pre-wrap"></pre>
{erd_legend_html(mermaid_text)}
<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
const source = {json.dumps(mermaid_text)};
mermaid.initialize({{ startOnLoad: false, theme: "default" }});
async function draw(text) {{
  const {{ svg }} = await mermaid.render("erd-" + Math.random().toString(36).slice(2), text);
  document.getElementById("erd").innerHTML = svg;
}}
try {{
  await draw(source);
}} catch (error) {{
  try {{
    await draw(source.split("\\n").filter(line => !/^\\s*(classDef|class) /.test(line)).join("\\n"));
  }} catch (error2) {{
    const fallback = document.getElementById("erd-fallback");
    fallback.textContent = source + "\\n\\n" + error2;
    fallback.style.display = "block";
  }}
}}
</script>"""
    displayHTML(html)

In [0]:
# ------------------------------------------------------------
# docs/silver_data_model.md: whole silver model, regenerated from the contracts
# (GitHub/GitLab render the mermaid block natively)
# ------------------------------------------------------------
def write_data_model_doc(contracts=None):
    contracts = contracts if contracts is not None else load_all_entity_contracts()
    roles = ", ".join(info["label"] for info in erd_role_styles.values())
    lines = [
        "# Silver data model",
        "",
        f"_Generated from `{entity_contracts_folder}/` by `write_data_model_doc`. Do not edit by hand._",
        "",
        f"- **Table roles** (colours in the diagram): {roles}.",
        "- **Entities keep the records of every data source:** the PK includes `source_name`. "
        "Resolving several sources into one golden record is the job of MDM (future `<entity>_golden` tables).",
        f"- **Every table references `{data_source_entity}`** through `{fk_column_name(data_source_entity)}`.",
        "- **Naming:** \"source\" always means data source. Energy technologies are a business column "
        "called `technology`.",
        "",
        "```mermaid",
        build_erd_mermaid(contracts),
        "```",
        "",
        "## Data sources",
        "",
        "| source_name | source_type | provider | license |",
        "|---|---|---|---|",
    ]
    for row in build_data_source_rows(contracts):
        lines.append(f"| `{row['source_name']}` | {row['source_type']} | {row.get('provider') or '—'} | "
                     f"{row.get('license') or '—'} |")
    lines += ["", "## Tables", ""]

    for name in sorted(contracts):
        contract = contracts[name]
        for table in build_silver_tables(contract):
            pk = table["primary_key"]
            grain = contract["grain"] + (" and data source" if table["role"] == "entity" else "")
            lines.append(f"### `{silver_table_full_name(table['name'])}`")
            lines.append(f"- **Entity:** `{name}` ({contract['domain']}) · role: {table['role']}")
            lines.append(f"- **Description:** {contract['description']}")
            lines.append(f"- **Grain:** {grain}")
            lines.append(f"- **Primary key:** `{pk['name']}` = md5({', '.join(pk['columns'])})")
            for fk in table["foreign_keys"]:
                lines.append(f"- **Foreign key:** `{fk['name']}` = {fk_hash_text(fk)} → "
                             f"`{fk['references_table']}.{fk['references_column']}`")
            lines.append("")

    path = os.path.join(find_repo_folder(docs_folder), data_model_doc_name)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path